# AIA frame-step variants: does the asymmetric frame step bias `delta`?

`phase/aia.py`'s two alternating steps do not minimize the same cost. Writing
the model over the `(N, P)` stack (`N` frames, `P = H*W` pixels) as

```
M = 1_N a^T  +  p u^T  +  q v^T        p_n = g_n cos(delta_n),  q_n = g_n sin(delta_n)
```

- **Pixel step** (`aia.py:452-455`) fixes `p, q` and minimizes `||I - M||^2`
  over `(a, u, v)`. Exact.
- **Frame step** (`aia.py:458-464`) *should* fix `(a, u, v)` and minimize over
  `(p, q)` -- design matrix `[a, u, v]` with the coefficient on `a` pinned to
  1. Instead it regresses onto `[1, u, v]` with a **free** per-frame
  coefficient `alpha_n`: the known per-pixel background `a` is replaced by an
  unknown per-frame scalar.

By linearity, regressing `a + p_n*u + q_n*v` onto `[1, u, v]` gives
`Phat_n = p_n + beta_u`, `Qhat_n = q_n + beta_v`, where `(beta_u, beta_v)` are
the `u, v` coefficients of `a` regressed on `[1, u, v]` **across pixels** --
the *same* offset for every frame. So

```
delta_hat_n = atan2(g_n sin(delta_n) + beta_v,  g_n cos(delta_n) + beta_u)
            ~= delta_n + (|beta|/g_n) * sin(delta_n - psi)
```

a first-harmonic error in the phase steps -- the textbook source of a **2*phi**
ripple in the recovered phase. Pinning `delta[0] = 0` does not remove it (a
translation is not a rotation). `beta -> 0` only when `a` is uncorrelated with
`u = b*cos(phi)`, `v = -b*sin(phi)` over the field, i.e. good phase coverage --
which is exactly what `kappa_ps` (`aia.py:492-498`) already measures.

`aia.py:334-341` records that subtracting `a` was tried and found empirically
*less* robust (its mid-iteration estimate carries its own leakage, and
subtracting it feeds that error back into the next pixel step). That's
plausible, but the free-`alpha` form's own bias `beta` was never measured. This
notebook measures it on real data and compares four frame-step variants head
to head. **No changes are made to `phase/aia.py`** -- the loop is
reimplemented below and gated against `phase.aia.aia` for exactness (Section 3).

## The four variants

All four share identical `delta0`, identical `g`, and an identical pixel step.
Only the frame-step design matrix differs.

| key | frame-step regression | unknowns/frame | what it is |
|---|---|---|---|
| `free_alpha` | `I_n ~ [1, u, v]` | `alpha_n, P_n, Q_n` | **current code** (`aia.py:458-464`) |
| `subtract_a` | `(I_n - a) ~ [u, v]` | `P_n, Q_n` | exact block-coordinate descent; monotone joint cost guaranteed |
| `subtract_a_dc` | `(I_n - a) ~ [1, u, v]` | `alpha_n, P_n, Q_n` | subtract `a`, but keep a free DC term for per-frame brightness drift |
| `free_ahat` | `I_n ~ [1, centered(a), u, v]` | 4 | `a`'s own coefficient is free -- used where it helps, shrunk to 0 where it's wrong |

`subtract_a` vs `subtract_a_dc` is the factorial control that separates "does
subtracting `a` matter" from "does the free DC column matter".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from types import SimpleNamespace

from phase import remove_carrier, subtract_reference, combine_acquisitions, estimate_phase_ripple
from phase.aia import aia as aia_reference, measure_frame_contrast, _cond3, _chunked_sigma
from phase.backend import get_array_module, to_device, wrap, wrap_sub, asnumpy
from phase import backend as _backend

REPO_ROOT = "/Users/rodionsa/Library/CloudStorage/GoogleDrive-sergeyrodionov93@gmail.com/My Drive/work/Projects/HoloEML/holoeml-processing"
DATA_DIR = REPO_ROOT + "/data/"

delta_crop = 400  # -> (30, 2200, 3296) per stack, ~870 MB float32; same crop as the other notebooks

# independent bare-glass pair -- true difference is 0, so the residual after
# subtract_reference + remove_carrier is pure error (Section 9)
NULL_FILES = [
    "piezo_scan_empty_20260721_174643.npz",
    "piezo_scan_empty_20260721_174706.npz",
]
# all 5 empty repeats, for the optional repeatability sweep (Section 10)
EMPTY_ALL = [
    "piezo_scan_empty_20260721_174643.npz",
    "piezo_scan_empty_20260721_174706.npz",
    "piezo_scan_empty_20260721_174728.npz",
    "piezo_scan_empty_20260721_174753.npz",
    "piezo_scan_empty_20260721_174815.npz",
]

GAIN_MODE = "auto"   # g measured once per stack (measure_frame_contrast), shared by all 4 variants
ITERS, TOL = 30, 1e-4
RUN_SLOW = False     # gates Section 10 (5-repeat scatter, ~10 more aia_variant calls per mode)

MODES = ["free_alpha", "subtract_a", "subtract_a_dc", "free_ahat"]
COLORS = {"free_alpha": "C0", "subtract_a": "C1", "subtract_a_dc": "C2", "free_ahat": "C3"}

## `aia_variant`: `phase.aia.aia`'s loop with a switchable frame step

A line-by-line mirror of `aia()` (`phase/aia.py:422-471`) -- same pixel step,
same buffer-reuse discipline, same diagnostics -- except the frame-step design
matrix is chosen by `mode`. `mode="free_alpha"` uses the exact same
preallocated-`F`-order-buffer pattern as the shipped code (not `column_stack`
each iteration), specifically so it reproduces `aia()` bit-for-bit -- checked
in Section 3 below. `_cond3`, `_chunked_sigma`, `measure_frame_contrast` are
reused directly from `phase.aia` rather than reimplemented.

Also records, per iteration:
- `beta_history`: `(beta_u, beta_v)`, the `[u, v]` coefficients of the current
  `a` regressed on `[1, u, v]` across pixels -- assembled from 5 scalar
  reductions, the same pattern `aia.py` already uses for `kappa_ps`
  (`aia.py:492-498`), so it costs O(P) with no extra `(P, 3)` allocation.
- `joint_cost_history` (only if `track_cost=True`): the RMS of
  `I - (1 a^T + p u^T + q v^T)` on the *exact* bilinear model, via
  `_chunked_sigma` -- the one yardstick all four modes can be compared on,
  since they fit different frame-step models. Left off by default (an extra
  full pass over `I` per iteration) -- turn on only for the single-stack
  diagnostics in Sections 4-8, not the multi-file loops in Sections 9-10.

In [ ]:
def aia_variant(stack, mode, delta0=None, iters=30, tol=1e-4, gain=None,
                 device="auto", dtype=None, track_cost=False):
    """phase.aia.aia's alternating solve with a switchable frame-step design.

    mode:
      "free_alpha"    I_n ~ [1, u, v]                  current phase.aia.aia
      "subtract_a"    (I_n - a) ~ [u, v]                exact block-coordinate descent
      "subtract_a_dc" (I_n - a) ~ [1, u, v]             subtract a, free per-frame DC
      "free_ahat"     I_n ~ [1, centered(a), u, v]      a's own coefficient free

    Returns a SimpleNamespace with the same fields as AIAResult plus
    delta_history (iters_run+1, N), beta_history (iters_run, 2), and
    joint_cost_history (iters_run,) if track_cost.
    """
    if mode not in ("free_alpha", "subtract_a", "subtract_a_dc", "free_ahat"):
        raise ValueError(f"unknown mode {mode!r}")

    stack = to_device(stack, device=device)
    xp = get_array_module(stack)
    N, H, W = stack.shape
    work_dtype = dtype if dtype is not None else _backend.default_dtype(xp)
    I = stack.reshape(N, -1).astype(work_dtype, copy=False)        # (N, P)
    P = I.shape[1]

    if delta0 is None:
        delta0 = xp.arange(N) * 2 * xp.pi / N
    delta = xp.asarray(delta0, dtype=xp.float64).copy()

    if gain is None:
        g = xp.ones(N, dtype=xp.float64)
    elif isinstance(gain, str) and gain == "auto":
        g = measure_frame_contrast(stack, dtype=work_dtype)
    else:
        g = xp.asarray(gain, dtype=xp.float64)

    # frame-step design buffer, width/columns depend on mode -- preallocated
    # once and overwritten in place each iteration, same discipline as
    # aia.py:443-444 (the "free_alpha" branch below is byte-for-byte the same
    # sequence of ops as aia()'s frame step).
    if mode in ("free_alpha", "subtract_a_dc"):
        B = xp.empty((P, 3), dtype=work_dtype, order="F")
        B[:, 0] = 1
    elif mode == "subtract_a":
        B = xp.empty((P, 2), dtype=work_dtype, order="F")
    else:  # free_ahat
        B = xp.empty((P, 4), dtype=work_dtype, order="F")
        B[:, 0] = 1

    u = v = a = None
    A = None
    converged = False
    it = 0
    delta_history = [asnumpy(delta).copy()]
    beta_history = []
    joint_cost_history = []

    for it in range(iters):
        # pixel step: identical to aia.py:452-455 for every mode
        c, s = xp.cos(delta), xp.sin(delta)
        A = xp.column_stack([xp.ones(N), g * c, g * s])                # (N,3) float64
        X = xp.linalg.pinv(A).astype(work_dtype) @ I                   # (3,P)
        a, u, v = X[0], X[1], X[2]

        # beta diagnostic: (u,v) coefficients of a ~ [1,u,v] across pixels
        Su, Sv = float(xp.sum(u)), float(xp.sum(v))
        Suu, Svv, Suv = float(xp.sum(u * u)), float(xp.sum(v * v)), float(xp.sum(u * v))
        Sa, Sau, Sav = float(xp.sum(a)), float(xp.sum(a * u)), float(xp.sum(a * v))
        try:
            beta_u, beta_v = np.linalg.solve(
                np.array([[float(P), Su, Sv], [Su, Suu, Suv], [Sv, Suv, Svv]]),
                np.array([Sa, Sau, Sav]),
            )[1:]
        except np.linalg.LinAlgError:
            beta_u, beta_v = np.nan, np.nan
        beta_history.append((float(beta_u), float(beta_v)))

        # frame step: design matrix and target depend on mode
        if mode == "free_alpha":
            B[:, 1], B[:, 2] = u, v
            BtB = (B.T @ B).astype(xp.float64)
            IB = (I @ B).astype(xp.float64)
            x = xp.linalg.solve(BtB, IB.T)
            Pn, Qn = x[1], x[2]
        elif mode == "subtract_a":
            B[:, 0], B[:, 1] = u, v
            BtB = (B.T @ B).astype(xp.float64)
            aB = (a.astype(xp.float64) @ B.astype(xp.float64))          # (2,)
            IB = (I @ B).astype(xp.float64) - aB[None, :]
            x = xp.linalg.solve(BtB, IB.T)
            Pn, Qn = x[0], x[1]
        elif mode == "subtract_a_dc":
            B[:, 1], B[:, 2] = u, v
            BtB = (B.T @ B).astype(xp.float64)
            aB = (a.astype(xp.float64) @ B.astype(xp.float64))          # (3,)
            IB = (I @ B).astype(xp.float64) - aB[None, :]
            x = xp.linalg.solve(BtB, IB.T)
            Pn, Qn = x[1], x[2]
        else:  # free_ahat
            B[:, 1] = a - a.mean()
            B[:, 2], B[:, 3] = u, v
            BtB = (B.T @ B).astype(xp.float64)
            IB = (I @ B).astype(xp.float64)
            x = xp.linalg.pinv(BtB) @ IB.T   # pinv: centered-a column can be near-degenerate
            Pn, Qn = x[2], x[3]

        new_delta = xp.arctan2(Qn, Pn)
        new_delta = new_delta - new_delta[0]                          # pin phase origin
        step = float(xp.abs(wrap(new_delta - delta)).max())
        delta = new_delta
        delta_history.append(asnumpy(delta).copy())

        if track_cost:
            joint_cost_history.append(_chunked_sigma(I, A, xp.vstack([a, u, v]), xp))

        if step < tol:
            converged = True
            break

    phi = xp.arctan2(-v, u).reshape(H, W)
    u64, v64 = u.astype(xp.float64), v.astype(xp.float64)
    b = xp.sqrt(u64 ** 2 + v64 ** 2).reshape(H, W)
    a_map = a.reshape(H, W)

    kappa_p = _cond3(A.T @ A, xp)

    r = xp.maximum(xp.sqrt(u64 ** 2 + v64 ** 2), xp.finfo(xp.float64).eps)
    cphi, sphi = u64 / r, -v64 / r
    Scp, Ssp = float(cphi.sum()), float(sphi.sum())
    Scc, Sss = float((cphi * cphi).sum()), float((sphi * sphi).sum())
    Scs = float((cphi * sphi).sum())
    CtC = xp.asarray([[float(P), Scp, Ssp], [Scp, Scc, Scs], [Ssp, Scs, Sss]])
    kappa_ps = _cond3(CtC, xp)

    sigma = _chunked_sigma(I, A, xp.vstack([a, u, v]), xp)
    b_amp = max(float(xp.median(b)), np.finfo(float).eps)
    predicted_rms = 0.42 * (np.sqrt(kappa_p) + 2) * (sigma / b_amp) / np.sqrt(N)

    return SimpleNamespace(
        mode=mode, phi=phi, b=b, a=a_map, delta=delta, g=g,
        kappa_p=kappa_p, kappa_ps=kappa_ps, predicted_rms=predicted_rms,
        iters_run=it + 1, converged=converged,
        delta_history=np.array(delta_history),
        beta_history=np.array(beta_history),
        joint_cost_history=np.array(joint_cost_history),
    )

## Section 3 -- validation gate

`aia_variant(..., mode="free_alpha")` must reproduce `phase.aia.aia` --
everything downstream is meaningless otherwise. Run both on the same crop
with the same `gain` and compare `delta` and `phi` directly. Tolerance is set
at machine-precision scale (not `1e-12` on the nose): the two implementations
do the identical sequence of BLAS calls, but `phase.aia.aia`'s `pinv`/`solve`
calls happen in a different Python-level call stack, which can select a
different (still IEEE-754-valid) summation order inside the BLAS routine --
a real (if tiny) floating-point effect, not a bug in either implementation.

In [ ]:
data0 = np.load(DATA_DIR + NULL_FILES[0])
stack0 = data0["images"][:, delta_crop:3000 - delta_crop, delta_crop:4096 - delta_crop, 0]
print("stack0 shape:", stack0.shape, stack0.dtype)

ref_result = aia_reference(stack0, iters=ITERS, tol=TOL, gain=GAIN_MODE)
var_result = aia_variant(stack0, mode="free_alpha", iters=ITERS, tol=TOL, gain=GAIN_MODE, track_cost=True)

max_ddelta_deg = np.degrees(np.abs(asnumpy(ref_result.delta) - asnumpy(var_result.delta)).max())
max_dphi_deg = np.degrees(np.abs(asnumpy(wrap_sub(ref_result.phi, var_result.phi))).max())

print(f"converged: ref={ref_result.converged} ({ref_result.iters_run} it), "
      f"variant={var_result.converged} ({var_result.iters_run} it)")
print(f"max |delta_ref - delta_free_alpha| = {max_ddelta_deg:.3e} deg")
print(f"max |phi_ref - phi_free_alpha|     = {max_dphi_deg:.3e} deg")

assert max_ddelta_deg < 1e-6, "aia_variant(free_alpha) does not reproduce phase.aia.aia -- stop here"
print("PASS: aia_variant(mode='free_alpha') reproduces phase.aia.aia")

## Section 4 -- the `beta` diagnostic and its falsifiable prediction

`beta = (beta_u, beta_v)` at convergence is the leakage of `a` into the
frame-step's `[u, v]` columns. If the free-`alpha` bias mechanism described
above is real, then `delta_free_alpha - delta_subtract_a` should equal the
*exact* (non-linearized) prediction

```
predicted_ddelta_n = atan2(q_n + beta_v, p_n + beta_u) - delta_n
```

evaluated at `subtract_a`'s own converged `(p_n, q_n, delta_n)` and
`free_alpha`'s converged `beta`. This cell runs `subtract_a` (the
unbiased reference) and overlays predicted vs. measured.

In [ ]:
result_subtract_a = aia_variant(stack0, mode="subtract_a", iters=ITERS, tol=TOL, gain=GAIN_MODE, track_cost=True)

beta_u, beta_v = var_result.beta_history[-1]
beta_mag = float(np.hypot(beta_u, beta_v))
g = asnumpy(var_result.g)
g_med = float(np.median(g))
print(f"beta_u = {beta_u:.6g}, beta_v = {beta_v:.6g}, |beta| = {beta_mag:.6g}")
# beta is a regression coefficient on the SAME [u, v] predictors that P_n, Q_n
# (= g_n*cos/sin(delta_n)) are fit against -- so it lives on g's O(1) scale, not
# b's O(counts) scale (b = sqrt(u^2+v^2) is itself in intensity units). Normalize
# by median(g), not median(b).
print(f"median(g) = {g_med:.4f}    |beta| / median(g) = {beta_mag / g_med:.4f} rad "
      f"= {np.degrees(beta_mag / g_med):.4f} deg  (linearized delta bias)")

delta_sub = asnumpy(result_subtract_a.delta)
p_n = g * np.cos(delta_sub)
q_n = g * np.sin(delta_sub)
predicted_ddelta = np.degrees(wrap(np.arctan2(q_n + beta_v, p_n + beta_u) - delta_sub))
measured_ddelta = np.degrees(wrap(asnumpy(var_result.delta) - delta_sub))

corr = np.corrcoef(predicted_ddelta, measured_ddelta)[0, 1]
print(f"predicted peak |ddelta| = {np.abs(predicted_ddelta).max():.4f} deg")
print(f"measured  peak |ddelta| = {np.abs(measured_ddelta).max():.4f} deg")
print(f"correlation(predicted, measured) across frames = {corr:.4f}")

order = np.argsort(delta_sub)
plt.figure(figsize=(7, 5))
plt.plot(np.degrees(delta_sub[order]), predicted_ddelta[order], "o-", label="predicted from beta")
plt.plot(np.degrees(delta_sub[order]), measured_ddelta[order], "x--", label="measured (free_alpha - subtract_a)")
plt.xlabel("delta_n (deg, subtract_a reference)")
plt.ylabel("delta_free_alpha - delta_subtract_a (deg)")
plt.title("Predicted vs. measured free-alpha bias in delta")
plt.legend()
plt.grid(True)
plt.tight_layout()

## Section 5 -- `delta` across all four variants

Run the remaining two variants, then compare all four `delta` vectors
(referenced to `delta[0] = 0`) plus convergence/conditioning diagnostics, and
check each against the piezo's own `positions` log (nominally linear in
`delta`, though piezo hysteresis can bend this curve independently of any AIA
error -- a partial, not fully independent, check).

In [ ]:
result_subtract_a_dc = aia_variant(stack0, mode="subtract_a_dc", iters=ITERS, tol=TOL, gain=GAIN_MODE, track_cost=True)
result_free_ahat = aia_variant(stack0, mode="free_ahat", iters=ITERS, tol=TOL, gain=GAIN_MODE, track_cost=True)

results = {
    "free_alpha": var_result,
    "subtract_a": result_subtract_a,
    "subtract_a_dc": result_subtract_a_dc,
    "free_ahat": result_free_ahat,
}

print(f"{'mode':<15} {'converged':<10} {'iters':<6} {'kappa_p':<10} {'kappa_ps':<10} {'pred_rms(deg)':<14}")
for m in MODES:
    r = results[m]
    print(f"{m:<15} {str(r.converged):<10} {r.iters_run:<6} {r.kappa_p:<10.3f} {r.kappa_ps:<10.3f} "
          f"{np.degrees(r.predicted_rms):<14.4f}")

In [ ]:
plt.figure(figsize=(7, 5))
for m in MODES:
    d = np.degrees(asnumpy(results[m].delta))
    plt.plot(d, ".-", label=m, color=COLORS[m])
plt.xlabel("frame n")
plt.ylabel("delta_n (deg)")
plt.title("Converged phase steps, all four variants")
plt.legend()
plt.grid(True)
plt.tight_layout()

print("pairwise max |delta_i - delta_j| (deg):")
for i, mi in enumerate(MODES):
    for mj in MODES[i + 1:]:
        dd = np.degrees(np.abs(wrap(asnumpy(results[mi].delta) - asnumpy(results[mj].delta))).max())
        print(f"  {mi:<15} vs {mj:<15} {dd:.4f} deg")

In [ ]:
positions = data0["volts"] if "volts" in data0.files else data0["positions"]
plt.figure(figsize=(7, 5))
for m in MODES:
    d = np.degrees(asnumpy(results[m].delta))
    fit = np.polyfit(positions, d, 1)
    resid = d - np.polyval(fit, positions)
    plt.plot(positions, resid, "o-", label=f"{m} (std={resid.std():.3f} deg)", color=COLORS[m])
plt.xlabel("piezo position / volts (as logged)")
plt.ylabel("delta_n - linear fit (deg)")
plt.title("Residual from a straight-line delta-vs-piezo fit\n(piezo hysteresis can also bend this -- partial check only)")
plt.legend()
plt.grid(True)
plt.tight_layout()

## Section 6 -- convergence of the joint cost

`joint_cost_history` is the RMS residual of the *exact* bilinear model
`I - (1 a^T + p u^T + q v^T)`, evaluated identically for all four modes (via
`_chunked_sigma`) even though each mode's frame step optimizes something
different. `subtract_a` is true block-coordinate descent on this cost, so it
must be monotone non-increasing; the others have no such guarantee -- this
cell shows whether that guarantee is actually violated on real data.

In [ ]:
plt.figure(figsize=(7, 5))
for m in MODES:
    J = results[m].joint_cost_history
    plt.plot(np.arange(1, len(J) + 1), J, ".-", label=m, color=COLORS[m])
plt.xlabel("iteration")
plt.ylabel("joint cost (RMS of I - model, work_dtype units)")
plt.yscale("log")
plt.title("Joint-cost convergence -- subtract_a must be monotone")
plt.legend()
plt.grid(True, which="both")
plt.tight_layout()

for m in MODES:
    J = results[m].joint_cost_history
    n_increases = int(np.sum(np.diff(J) > 0))
    print(f"{m:<15} increases in {n_increases}/{len(J) - 1} steps "
          f"(final={J[-1]:.6g})")

## Section 7 -- spatial pattern of the disagreement in `phi`

If the free-`alpha` bias is the first-harmonic-in-`delta` mechanism from the
intro, the disagreement between variants should look like fringes (locked to
the carrier), not spatially random noise.

In [ ]:
carrier_kwargs = dict(defocus=True, refine_iters=10, n_blocks=10)
phi_clean = {m: remove_carrier(results[m].phi, results[m].b, **carrier_kwargs).phi for m in MODES}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, m in zip(axes, ["subtract_a", "subtract_a_dc", "free_ahat"]):
    diff_deg = np.degrees(asnumpy(wrap_sub(phi_clean[m], phi_clean["free_alpha"])))
    im = ax.imshow(diff_deg, cmap="RdBu_r", vmin=-np.nanpercentile(np.abs(diff_deg), 99),
                    vmax=np.nanpercentile(np.abs(diff_deg), 99))
    ax.set_title(f"{m} - free_alpha\nstd={diff_deg.std():.3f} deg, p99={np.percentile(np.abs(diff_deg), 99):.3f} deg")
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()

## Section 8 -- ripple spectrum `eps(phi)`

The direct signature: `estimate_phase_ripple` bins the post-carrier-removal
residual by wrapped phase and fits a low-order Fourier series. A
first-harmonic error in `delta_n` predicts a **2*phi** (order-2) term in
`eps(phi)` -- that's the coefficient to watch.

In [ ]:
b_thresh = {m: float(np.median(asnumpy(results[m].b))) * 0.3 for m in MODES}
ripples = {
    m: estimate_phase_ripple(
        results[m].phi,
        mask=asnumpy(results[m].b) > b_thresh[m],
        weight=results[m].b,
        orders=(1, 2, 3, 4),
    )
    for m in MODES
}

plt.figure(figsize=(8, 5))
for m in MODES:
    rr = ripples[m]
    plt.plot(np.degrees(asnumpy(rr.bin_centers)), np.degrees(asnumpy(rr.lut)), ".", label=m,
             color=COLORS[m], alpha=0.6)
plt.xlabel("wrapped phase phi (deg)")
plt.ylabel("binned residual eps(phi) (deg)")
plt.title("Phase-locked ripple, all four variants")
plt.legend()
plt.grid(True)
plt.tight_layout()

print(f"{'mode':<15} {'rms_before':<12} {'rms_after':<12} " + " ".join(f"order{k}(deg)" for k in (1, 2, 3, 4)))
for m in MODES:
    rr = ripples[m]
    amps = [np.degrees(np.hypot(*rr.coeffs[k])) for k in (1, 2, 3, 4)]
    print(f"{m:<15} {np.degrees(rr.rms_before):<12.4f} {np.degrees(rr.rms_after):<12.4f} " +
          " ".join(f"{a:<11.4f}" for a in amps))

## Section 9 -- null test (decisive metric)

Same protocol as the findings already recorded in
`scripts/test/test_phase_difference.ipynb` (that notebook's cell 9 markdown):
two independent scans of the same nothing, so the true difference is 0 and
anything left after `subtract_reference` + `remove_carrier` is pure error.
Recorded baselines there: **2.68 deg / 5.63 deg** (std/p99) at `gain=None`,
**~1.3-1.5 deg / ~4-4.2 deg** at `gain="auto"` (printed below for
orientation -- from a different scan pair, so treat as context, not a strict
apples-to-apples number).

Loads one stack at a time and keeps only `phi`/`b` per variant, so peak memory
stays at roughly one stack (~870 MB) plus 4 variants' `phi`+`b`
(4 * 7.25M px * 4 B * 2 maps ~ 232 MB), not two full stacks at once.

In [ ]:
def load_crop(fname):
    d = np.load(DATA_DIR + fname)
    return d["images"][:, delta_crop:3000 - delta_crop, delta_crop:4096 - delta_crop, 0]

null_phi = {m: [] for m in MODES}
null_b = {m: [] for m in MODES}

for fname in NULL_FILES:
    stack = load_crop(fname)
    for m in MODES:
        r = aia_variant(stack, mode=m, iters=ITERS, tol=TOL, gain=GAIN_MODE, track_cost=False)
        null_phi[m].append(r.phi)
        null_b[m].append(r.b)
        if not r.converged:
            print(f"WARNING: {fname} / {m} did not converge in {ITERS} iters")
    del stack

In [ ]:
null_std_deg, null_p99_deg = {}, {}
for m in MODES:
    diff = subtract_reference(null_phi[m][0], null_phi[m][1], null_b[m][0])
    clean = remove_carrier(diff.phi, null_b[m][0], **carrier_kwargs).phi
    clean_deg = np.degrees(asnumpy(clean))
    null_std_deg[m] = float(clean_deg.std())
    null_p99_deg[m] = float(np.percentile(np.abs(clean_deg), 99))

print(f"{'mode':<15} {'std(deg)':<10} {'p99(deg)':<10}")
for m in MODES:
    print(f"{m:<15} {null_std_deg[m]:<10.4f} {null_p99_deg[m]:<10.4f}")
print()
print("recorded baselines (test_phase_difference.ipynb, different scan pair):")
print("  gain=None : std=2.68 deg, p99=5.63 deg")
print("  gain=auto : std~1.3-1.5 deg, p99~4-4.2 deg")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(MODES, [null_std_deg[m] for m in MODES], color=[COLORS[m] for m in MODES])
ax1.set_ylabel("std (deg)")
ax1.set_title("Null-test residual std")
ax1.tick_params(axis="x", rotation=20)
ax2.bar(MODES, [null_p99_deg[m] for m in MODES], color=[COLORS[m] for m in MODES])
ax2.set_ylabel("p99 (deg)")
ax2.set_title("Null-test residual p99")
ax2.tick_params(axis="x", rotation=20)
plt.tight_layout()

## Section 10 -- repeatability across 5 repeats (optional, `RUN_SLOW`)

Separates random scatter (this section) from bias (Section 9 is the only
section that can see bias, since only a null test has a known-zero answer).
Set `RUN_SLOW = True` in the config cell to run this -- it's `5 files x 4
modes = 20` more `aia_variant` calls at full ROI.

In [ ]:
if RUN_SLOW:
    repeat_phi = {m: [] for m in MODES}
    repeat_b = {m: [] for m in MODES}
    for fname in EMPTY_ALL:
        stack = load_crop(fname)
        for m in MODES:
            r = aia_variant(stack, mode=m, iters=ITERS, tol=TOL, gain=GAIN_MODE, track_cost=False)
            repeat_phi[m].append(r.phi)
            repeat_b[m].append(r.b)
        del stack

    print(f"{'mode':<15} {'median scatter (deg)':<20}")
    for m in MODES:
        combined = combine_acquisitions(repeat_phi[m], repeat_b[m])
        med_scatter_deg = float(np.degrees(np.median(asnumpy(combined.scatter))))
        print(f"{m:<15} {med_scatter_deg:<20.4f}")
else:
    print("RUN_SLOW is False -- skipped. Set RUN_SLOW = True in the config cell to run this section.")

## Section 11 -- results and decision rule

Fill in from the cells above:

| metric | free_alpha | subtract_a | subtract_a_dc | free_ahat |
|---|---|---|---|---|
| `\|beta\| / median(g)` (Sec. 4, linearized bias, deg) | -- | -- | -- | -- |
| null-test std / p99 (deg) (Sec. 9) | | | | |
| order-2 ripple amplitude (deg) (Sec. 8) | | | | |
| monotone joint cost? (Sec. 6) | | | | |
| repeat scatter (deg) (Sec. 10, if run) | | | | |

**Decision rule:** change `phase/aia.py`'s frame step only if a variant beats
`free_alpha` on the Section 9 null-test std, *and* reduces the Section 8
order-2 amplitude, *and* the Section 4 predicted/measured curves visibly
agree (confirming the mechanism, not just correlating with it).

If `|beta| / median(g)` is negligible and all four `delta` vectors agree well
under the ~1.15 deg/scan noise floor already documented for this setup, the
asymmetry is real but harmless *on this data* -- the current formulation is
vindicated, and `aia.py:334-341`'s docstring should gain the measured `beta`
as evidence for why subtracting `a` was tried and rejected, rather than
leaving that claim unquantified.